In [1]:
import pickle
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "animakinetica").exists():
    ROOT = ROOT.parent
if not (ROOT / "animakinetica").exists():
    raise FileNotFoundError("Could not locate the workspace root containing 'animakinetica'.")

# Enter the export tag to include (for example, "p2", "s2", or "opt").
# Set to "all" to include every export.
EXPORT_FILTER = "p2".strip().casefold()
if not EXPORT_FILTER:
    raise ValueError("EXPORT_FILTER must be a non-empty export tag or 'all'.")

CAMPAIGNS = {
    "1": ROOT / "Campaign_1_fixed_stoichiometry_Humbird",
    "2": ROOT / "Campaign_6_fixed_stoichiometry_Humbird_enhanced",
    "3": ROOT / "Campaign_2_fixed_stoichiometry_GFI",
    # "3": ROOT / "Campaign_3_dynamic_stoichiometry_GFI",
    "4": ROOT / "Campaign_4_mode_comparison_excesskinetics",
    "5": ROOT / "Campaign_5_mode_comparison_adjustedexcesskinetics",
}

# (Model, Cell Properties) metadata per cell line
CAMPAIGN_INFO = {
    "1": ("Fixed", "Humbird (2020) Wildtype"),
    "2": ("Fixed", "Humbird (2020) Enhanced"),
    "3": ("Fixed", "Frohlich et al. (2025)"),
    # "3": ("Growth rate dependent", "Frohlich et al. (2025)"),
    "4": ("Substrate Dependent", "Frohlich et al. (2025)"),
    "5": ("Substrate Dependent Enhanced", "Frohlich et al. (2025)"),
}

# Metric columns (on a wet cell matter basis where applicable)
STY_COLUMN = "STY_WCM_kg_m3_day"
FCE_COLUMN = "feed_conversion_efficiency_kg_kg_WCM"
PCE_COLUMN = "protein_conversion_efficiency_kg_kg"
MCE_COLUMN = "medium_conversion_efficiency_m3_kg_WCM"

MODE_ABBREVIATIONS = {
    "Batch": "Batch",
    "Batch (startup Fedbatch)": "Batch",
    "Fedbatch": "FB",
    "Concentrated Fedbatch": "CF",
    "Concentrated FedBatch": "CF",
    "Continuous Perfusion": "CP",
    "Chemostat": "Chem",
}

STRATEGY_ABBREVIATIONS = {
    "constant": "Constant",
    "csfr": "CSFR",
    "exponential": "Exponential",
    "linear": "Linear",
    "gsp": "GSP",
    "cspr": "CSPR",
    "tbs": "TBS",
    "tbp": "TBP",
}

YIELD_LABELS = {
    "Glucose consumed": "Y<sub>Glc/X</sub>",
    "Glutamine consumed": "Y<sub>Gln/X</sub>",
    "Lactate produced": "Y<sub>Lac/X</sub>",
    "Ammonium produced": "Y<sub>NH4/X</sub>",
}

def load_simulation(path):
    with path.open("rb") as handle:
        return pickle.load(handle)

def matches_export_filter(path):
    """Return whether a pickle is inside a directory with the requested export tag."""
    if EXPORT_FILTER == "all":
        return True
    tag_pattern = re.compile(rf"(?:^|_){re.escape(EXPORT_FILTER)}(?:_|$)")
    return any(tag_pattern.search(directory.name.casefold()) for directory in path.parents)

def discover_simulation_candidates(campaign_dir):
    """Use pickle metadata, not possibly stale comparison CSV rows, for selection."""
    candidates = []
    for path in sorted(campaign_dir.rglob("*.pkl")):
        if not matches_export_filter(path):
            continue
        try:
            simulation = load_simulation(path)
            data = simulation.data if hasattr(simulation, "data") else simulation
            stats = data.summary_stats
            sty = pd.to_numeric(stats.get(STY_COLUMN), errors="coerce")
            fce = pd.to_numeric(stats.get(FCE_COLUMN), errors="coerce")
            pce = pd.to_numeric(stats.get(PCE_COLUMN), errors="coerce")
            mce = pd.to_numeric(stats.get(MCE_COLUMN), errors="coerce")
            if pd.notna(sty) and pd.notna(fce) and pd.notna(pce) and pd.notna(mce):
                candidates.append({
                    "run_id": path.stem,
                    "pickle": path,
                    "parent_name": path.parent.name,
                    "export_group": EXPORT_FILTER,
                    "simulation_type": getattr(data, "simulation_type", ""),
                    STY_COLUMN: float(sty),
                    FCE_COLUMN: float(fce),
                    PCE_COLUMN: float(pce),
                    MCE_COLUMN: float(mce),
                })
        except Exception as exc:
            print(f"Skipped unreadable pickle {path.name}: {exc}")
    if not candidates:
        raise FileNotFoundError(
            f"No valid {EXPORT_FILTER!r} simulation pickles found below {campaign_dir}"
        )
    return pd.DataFrame(candidates)

def format_simulation_name(row):
    simulation_type = str(row.get("simulation_type", "")).strip()
    parent_name = str(row.get("parent_name", ""))
    run_id = str(row.get("run_id", ""))
    full_id = f"{parent_name}_{run_id}".casefold()
    if simulation_type.casefold().startswith("batch") or full_id.startswith("batch_"):
        return "Batch"
    if simulation_type.casefold().startswith("chemostat") or full_id.startswith("chemostat_"):
        return "Chem"
    mode_abbr = MODE_ABBREVIATIONS.get(simulation_type, simulation_type)
    for token, label in sorted(STRATEGY_ABBREVIATIONS.items(), key=lambda item: len(item[0]), reverse=True):
        if re.search(rf"(?:^|_){re.escape(token)}(?:_|$)", full_id):
            return f"{mode_abbr} ({label})"
    return mode_abbr or "Unknown"

def apparent_stoichiometry(simulation):
    data = simulation.data if hasattr(simulation, "data") else simulation
    closure = data.compute_stoichiometric_closure()
    return {YIELD_LABELS[s["label"]]: float(s["yield_on_biomass"])
            for s in closure["component_specs"] if s["label"] in YIELD_LABELS}

rows = []
for cell_line, campaign_dir in CAMPAIGNS.items():
    candidates = discover_simulation_candidates(campaign_dir)
    print(f"Cell line {cell_line}: {len(candidates)} valid {EXPORT_FILTER!r} simulation exports found")
    selected = [
        ("+", candidates.loc[candidates[STY_COLUMN].idxmax()]),
        ("-", candidates.loc[candidates[STY_COLUMN].idxmin()]),
    ]
    model_name, cell_properties = CAMPAIGN_INFO[cell_line]
    for performance, row in selected:
        rows.append({
            "Cell line": cell_line,
            "Model": model_name,
            "Cell Properties": cell_properties,
            "Performance": performance,
            "Simulation": format_simulation_name(row),
            "STY (kg m⁻³ d⁻¹)": row[STY_COLUMN],
            "FCE (kg kg⁻¹)": row[FCE_COLUMN],
            "PCE (kg kg⁻¹)": row[PCE_COLUMN],
            "MCE (m³ kg⁻¹)": row[MCE_COLUMN],
            **apparent_stoichiometry(load_simulation(row["pickle"])),
        })

combined_table = pd.DataFrame(rows)
table_columns = [
    "Cell line",
    "Model",
    "Cell Properties",
    "Performance",
    "Simulation",
    "STY (kg m⁻³ d⁻¹)",
    "FCE (kg kg⁻¹)",
    "PCE (kg kg⁻¹)",
    "MCE (m³ kg⁻¹)",
    *YIELD_LABELS.values(),
]
combined_table = combined_table[table_columns]

formats = {
    "STY (kg m⁻³ d⁻¹)": "{:.2f}",
    "FCE (kg kg⁻¹)": "{:.2f}",
    "PCE (kg kg⁻¹)": "{:.2f}",
    "MCE (m³ kg⁻¹)": "{:.2f}",
    **{column: "{:.2f}" for column in YIELD_LABELS.values()},
}

paper_table = (
    combined_table.style.format(formats)
    .set_properties(**{
        "font-family": "Georgia, 'Times New Roman', serif",
        "font-size": "9pt",
        "color": "#000000",
        "background-color": "#ffffff",
        "border": "1px solid #7f7f7f",
        "padding": "6px 8px",
        "vertical-align": "middle",
    })
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("border-top", "1.5px solid #000"),
                ("border-bottom", "1.5px solid #000"),
                ("width", "100%"),
            ],
        },
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("text-align", "left"),
                ("font-family", "Georgia, 'Times New Roman', serif"),
                ("font-size", "10pt"),
                ("font-weight", "bold"),
                ("color", "#000"),
                ("padding-bottom", "6px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#f2f2f2"),
                ("color", "#000"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border-bottom", "1px solid #000"),
            ],
        },
        {"selector": "tbody tr:nth-child(even) td", "props": [("background-color", "#fafafa")]},
        {"selector": "tbody td:nth-child(1)", "props": [("text-align", "center"), ("font-weight", "bold")]},
        {"selector": "tbody td:nth-child(4)", "props": [("text-align", "center"), ("font-weight", "bold")]},
        {"selector": "tbody td:nth-child(n+6)", "props": [("text-align", "right"), ("font-variant-numeric", "tabular-nums")]},
    ])
    .hide(axis="index")
    .set_caption(
        f"Selected simulation performance and apparent stoichiometry by cell line ({EXPORT_FILTER.upper()} exports)"
    )
)
display(paper_table)

# output_path = ROOT / "Cross_Campaign_Comparison" / "export" / f"combined_simulation_stoichiometry_summary_{EXPORT_FILTER}.csv"
# combined_table.to_csv(output_path, index=False)
# print(f"Saved: {output_path}")

Cell line 1: 34 valid 'p2' simulation exports found
Cell line 2: 34 valid 'p2' simulation exports found
Cell line 3: 34 valid 'p2' simulation exports found
Cell line 4: 34 valid 'p2' simulation exports found
Cell line 5: 27 valid 'p2' simulation exports found


Cell line,Model,Cell Properties,Performance,Simulation,STY (kg m⁻³ d⁻¹),FCE (kg kg⁻¹),PCE (kg kg⁻¹),MCE (m³ kg⁻¹),YGlc/X,YGln/X,YLac/X,YNH4/X
1,Fixed,Humbird (2020) Wildtype,+,CP (Constant),18.79,2.74,4.23,0.22,0.33,0.05,0.47,0.07
1,Fixed,Humbird (2020) Wildtype,-,FB (CSFR),0.39,2.17,2.73,0.11,0.33,0.05,0.47,0.07
2,Fixed,Humbird (2020) Enhanced,+,CP (CSPR),74.73,0.84,1.66,0.05,0.15,0.02,0.04,0.00
2,Fixed,Humbird (2020) Enhanced,-,FB (CSFR),3.49,1.00,2.69,0.04,0.15,0.02,0.04,0.00
3,Fixed,Frohlich et al. (2025),+,CP (TBP),39.37,1.20,2.69,0.10,0.26,0.08,0.15,0.05
3,Fixed,Frohlich et al. (2025),-,FB (CSFR),1.15,1.71,8.82,0.04,0.26,0.08,0.15,0.05
4,Substrate Dependent,Frohlich et al. (2025),+,CP (CSPR),8.74,2.00,9.62,0.49,0.54,0.19,0.66,0.27
4,Substrate Dependent,Frohlich et al. (2025),-,FB (GSP),0.04,35.86,264.09,0.67,2.32,0.42,4.39,0.74
5,Substrate Dependent Enhanced,Frohlich et al. (2025),+,CP (TBP),42.56,0.82,2.77,0.09,0.32,0.06,0.07,0.03
5,Substrate Dependent Enhanced,Frohlich et al. (2025),-,FB (CSFR),1.07,3.55,22.19,0.14,0.44,0.09,0.29,0.07
